Étape 1 — SparkSession avec config s3a

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

spark = SparkSession.builder \
    .appName("TaaSim-NYC-ETL") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "20") \
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "taasim") \
    .config("spark.hadoop.fs.s3a.secret.key", "taasim123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ Spark prêt")

Cellule 2 — Lecture + nettoyage

In [ ]:
df_raw = spark.read.parquet(
    "s3a://raw/nyc-tlc/yellow_tripdata_2025-01.parquet",
    "s3a://raw/nyc-tlc/yellow_tripdata_2025-06.parquet",
    "s3a://raw/nyc-tlc/yellow_tripdata_2025-10.parquet"
)
print(f"Lignes brutes : {df_raw.count():,}")
print(f"Lignes brutes : {df_raw.count():,}")

df_clean = df_raw.filter(
    (F.col("trip_distance") > 0.1) & (F.col("trip_distance") <= 100) &
    (F.col("fare_amount") > 0) & (F.col("fare_amount") <= 300) &
    (F.col("passenger_count") >= 1) & (F.col("passenger_count") <= 9) &
    F.col("tpep_pickup_datetime").isNotNull() &
    (F.col("PULocationID") >= 1) & (F.col("PULocationID") <= 265)
).dropDuplicates(["tpep_pickup_datetime", "PULocationID", "total_amount"])

print(f"✅ Après nettoyage : {df_clean.count():,} lignes")

Cellule 3 — Mapping zones NYC → Casablanca

In [ ]:
mapping = spark.read.csv(
    "s3a://raw/nyc_to_casa_zone_mapping.csv",
    header=True, inferSchema=True
).select(
    F.col("locationid").cast(IntegerType()).alias("PULocationID_map"),
    F.col("arrondissement_id").cast(IntegerType()),
    F.col("arrondissement_name"),
    F.col("zone_type")
)

df_mapped = df_clean.join(
    F.broadcast(mapping),
    df_clean.PULocationID == F.col("PULocationID_map"),
    "left"
).drop("PULocationID_map") \
 .fillna({"arrondissement_id": 1,
           "arrondissement_name": "Maârif",
           "zone_type": "commercial"})

print(f"✅ Mapping appliqué")
df_mapped.groupBy("arrondissement_name").count().orderBy(F.desc("count")).show(5)

Cellule 4 — Remapping distance

In [ ]:
CASA_MAX_KM    = 18.0
CASA_MEDIAN_KM = 5.0

df_mapped = df_mapped.withColumn("distance_km_nyc", F.col("trip_distance") * 1.609)

w_pct = Window.orderBy("distance_km_nyc")
df_mapped = df_mapped.withColumn("distance_percentile", F.percent_rank().over(w_pct))

df_mapped = df_mapped.withColumn(
    "distance_casa_km",
    F.when(
        F.col("distance_percentile") <= 0.5,
        F.col("distance_percentile") * 2 * CASA_MEDIAN_KM
    ).otherwise(
        CASA_MEDIAN_KM + (F.col("distance_percentile") - 0.5) * 2 * (CASA_MAX_KM - CASA_MEDIAN_KM)
    )
)
print("✅ Distance remappée sur échelle Casablanca")

Cellule 5 — ETL complet → curated/

In [ ]:
df_agg = df_mapped.groupBy(
    F.col("arrondissement_id"),
    F.col("arrondissement_name"),
    F.col("zone_type"),
    F.hour("tpep_pickup_datetime").alias("hour_of_day"),
    F.dayofweek("tpep_pickup_datetime").alias("day_of_week"),
    F.when(F.dayofweek("tpep_pickup_datetime").isin([1,7]),1).otherwise(0).alias("is_weekend"),
    F.when(F.dayofweek("tpep_pickup_datetime")==6,1).otherwise(0).alias("is_friday"),
    F.date_format("tpep_pickup_datetime", "yyyy-MM").alias("year_month")
).agg(
    F.count("*").alias("trip_count"),
    F.avg("distance_casa_km").alias("avg_distance_casa_km"),
    F.avg("passenger_count").alias("avg_passengers"),
    F.percentile_approx("distance_casa_km", 0.5).alias("median_distance_km")
)

df_agg.write.mode("overwrite") \
    .partitionBy("year_month") \
    .parquet("s3a://curated/demand-by-zone/nyc/")

print(f"✅ ETL complet : {df_agg.count():,} lignes dans curated/demand-by-zone/nyc/")

Cellule 6 — Features ML → ml/features/

In [ ]:
w_lag = Window.partitionBy("arrondissement_id", "time_slot_30min", "day_of_week") \
              .orderBy("date_str")
w_roll = Window.partitionBy("arrondissement_id", "time_slot_30min") \
               .orderBy("date_str").rowsBetween(-6, 0)

df_features = df_mapped.withColumn(
    "time_slot_30min",
    (F.hour("tpep_pickup_datetime") * 60 + F.minute("tpep_pickup_datetime")) // 30
).withColumn(
    "date_str", F.date_format("tpep_pickup_datetime", "yyyy-MM-dd")
).groupBy(
    "arrondissement_id", "arrondissement_name", "zone_type",
    "time_slot_30min", "date_str",
    F.hour("tpep_pickup_datetime").alias("hour_of_day"),
    F.dayofweek("tpep_pickup_datetime").alias("day_of_week"),
    F.when(F.dayofweek("tpep_pickup_datetime").isin([1,7]),1).otherwise(0).alias("is_weekend"),
    F.when(F.dayofweek("tpep_pickup_datetime")==6,1).otherwise(0).alias("is_friday"),
    F.when((F.hour("tpep_pickup_datetime")>=7)&(F.hour("tpep_pickup_datetime")<=9),1).otherwise(0).alias("is_morning_peak"),
    F.when((F.hour("tpep_pickup_datetime")>=17)&(F.hour("tpep_pickup_datetime")<=19),1).otherwise(0).alias("is_evening_peak"),
).agg(
    F.count("*").alias("trip_count"),
    F.avg("distance_casa_km").alias("avg_distance_casa_km"),
).withColumn("demand_lag_1d", F.lag("trip_count", 1).over(w_lag)) \
 .withColumn("demand_lag_7d", F.lag("trip_count", 7).over(w_lag)) \
 .withColumn("rolling_7d_mean", F.avg("trip_count").over(w_roll)) \
 .fillna(0, subset=["demand_lag_1d", "demand_lag_7d"])

df_features.write.mode("overwrite") \
    .partitionBy("date_str") \
    .parquet("s3a://curated/nyc-ml-features/")

print(f"✅ Features ML : {df_features.count():,} lignes dans curated/nyc-ml-features/")

Cellule 7 — Vérification visuelle

In [ ]:
import matplotlib.pyplot as plt

pdf = df_agg.groupBy("hour_of_day") \
    .agg(F.avg("trip_count").alias("avg_demand")) \
    .orderBy("hour_of_day").toPandas()

plt.figure(figsize=(10, 4))
plt.bar(pdf["hour_of_day"], pdf["avg_demand"], color="#1D9E75")
plt.axvspan(7, 9, alpha=0.15, color="red", label="Pic matin")
plt.axvspan(17, 19, alpha=0.15, color="orange", label="Pic soir")
plt.title("Demande par heure — NYC remappé Casablanca")
plt.xlabel("Heure")
plt.ylabel("Courses moyennes")
plt.legend()
plt.tight_layout()
plt.savefig("/home/jovyan/work/image/nyc_demand_validation.png")
plt.show()
print("✅ Validation OK — sauvegarder ce graphique pour le rapport")